# What does a new cell cost?

This experiment asks whether a local boundary-event model can predict births per boundary read, including effects of later letters. See [the protocol](README.md) and [the append-only log](NOTEBOOK.md). The ordered-event recorder and stationary solver are calibrated. The excluded pilot failed its common-horizon gate; the main cohort remains unrun.

The immediate birth is fixed by the current turn and visited-neighbour mask. The unknown is their distribution. Births are not assumed independent regeneration points, and a pooled cross-rule transition matrix must respect target-rule-compatible successor states.

## Provenance and design
The following cell reads only committed design files under this experiment's results directory, verifies their hashes and checks assignment invariants. Counts below describe a planned cohort, not measured ants. It runs from the repository root or any descendant directory.

In [1]:
import csv, hashlib, json, subprocess
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Cargo.toml').exists() and (p / 'experiments').is_dir())
EXP = ROOT / 'experiments/2026-10-20-c-law-renewal'
inputs = [EXP / 'results/design/families.csv', EXP / 'results/design/provenance.json']
for p in inputs:
    subprocess.run(['git', 'ls-files', '--error-unmatch', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    subprocess.run(['git', '-c', 'core.filemode=false', 'diff', '--exit-code', 'HEAD', '--', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    print(hashlib.sha256(p.read_bytes()).hexdigest(), p.relative_to(ROOT))
provenance = json.loads(inputs[1].read_text())
assert hashlib.sha256(inputs[0].read_bytes()).hexdigest() == provenance['manifest_sha256']
print('Source revision:', provenance['base_revision'])
print('Notebook revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip())
print('Sampling seed:', provenance['sampling_seed'])
rows = list(csv.DictReader(inputs[0].open()))
families = defaultdict(list)
for row in rows:
    families[row['family']].append(row)
def necklace(rule):
    mirror = rule.translate(str.maketrans('RL', 'LR'))
    return min(r[i:] + r[:i] for r in (rule, mirror) for i in range(len(r)))
assert len({necklace(r['rule']) for r in rows}) == len(rows)
for family in families.values():
    assert len(family) == 3 and len({r['split'] for r in family}) == 1
    assert {r['arm'] for r in family} == {'base', 'early_swap', 'late_swap'}
    assert len({r['rule'].count('R') for r in family}) == 1
    for r in family:
        assert len(r['rule']) - len(r['rule'].lstrip('R')) == int(r['lead'])
print('Planned families by split:', dict(sorted(Counter(f[0]['split'] for f in families.values()).items())))
print('Planned rules:', len(rows))
print('This cell audits planned assignments only; pilot outcomes are audited below.')

bc37ff7e19aa4102dbbec69d17d39cb13986ea73f9225343263cad9eb1b946c8 experiments/2026-10-20-c-law-renewal/results/design/families.csv
46df2597f5e2fdab027d097513467be8f4c1913fd04667d219a9b8f34370b28a experiments/2026-10-20-c-law-renewal/results/design/provenance.json
Source revision: adf66774570469da30b10ffc1ceab911670a9d09


Notebook revision: e81f2385199b7e005df883bf0e3627b113740cd4
Sampling seed: 2026102001
Planned families by split: {'pilot': 4, 'train': 32, 'validation': 16}
Planned rules: 156
This cell audits planned assignments only; pilot outcomes are audited below.


## Method and controls
The source recorder is compared event by event to a sparse independent path oracle, plus exact totals against the existing aggregate tracer. The solver is calibrated on known two-state reward chains and challenged by hidden history. Rotor and random-walk controls use eight seeds each. The excluded pilot uses four families, three arms and two interior seeds, with horizons 2^22..2^26. Commands, seeds, thresholds and binary hashes are in the README and result provenance.

No main transition model is fitted. Draft P1–P4 remain untested; G1's solver calibration does not calibrate an ant state/backoff model. The support repeat at k=26 is a registered feasibility diagnostic after the horizon failure, not a rescue of the gate.

## What remains unestablished
There is no validated local renewal model, no derived intercept or slope, and no new mechanism claim about kappa. The pilot gate failed; no families were replaced and no validation outcomes were opened. Unseen-state mass is only a feasibility check, not a measure of predictive accuracy. Coarsening can improve support while destroying relevant information.

Next decisions must address both sampling failures and target-rule-compatible transitions. A good total birth rate alone cannot show Markov closure; the hidden-history calibration supplies a counterexample.

## Pilot provenance and results
Only committed JSON/CSV summaries are read. Every input is checked for content changes and hashed; ignored event dumps are not notebook inputs. Uncertainty below is between seeds, not a fit standard error.


In [2]:
files = [EXP / 'results' / name for name in ['calibration.json', 'pilot_report.json', 'pilot_diagnostics.json', 'pilot_diagnostics.csv']]
if (EXP / 'results/support_report.json').exists():
    files.append(EXP / 'results/support_report.json')
for p in files:
    subprocess.run(['git', 'ls-files', '--error-unmatch', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    subprocess.run(['git', '-c', 'core.filemode=false', 'diff', '--exit-code', 'HEAD', '--', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    print(hashlib.sha256(p.read_bytes()).hexdigest(), p.relative_to(ROOT))
calibration, controls, diagnostic = [json.loads(p.read_text()) for p in files[:3]]
support = json.loads(files[4].read_text()) if len(files) == 5 else None
assert calibration['passed']
print('Synthetic reward solver: maximum |c error|', calibration['max_absolute_error'], 'signed bias', calibration['signed_bias'], 'between-seed sd', calibration['between_seed_sd'])
print('Hidden-history control: pooled c', calibration['negative_control']['pooled_c'], 'conditional contrast', calibration['negative_control']['contrast'])
for walker, result in controls['controls'].items():
    assert result['passed'] and result['n'] == 8
    print(walker, 'c mean', result['mean'], 'between-seed sd', result['between_seed_sd'], 'escapes', result['escapes'])
assert controls['thread_check']['identical']
assert controls['horizon']['selected_log2'] is None
print('Original pilot horizon verdict:', controls['horizon']['status'])
print('Usable k=26 runs:', diagnostic['k26_usable_runs'], '/', diagnostic['k26_n'])
for name, result in (support['models'] if support else {}).items():
    print(name, 'unseen read mass median', result['median_unseen_mass'], 'maximum', result['max_unseen_mass'], 'runs >5%', result['runs_over_5pct'])
if support is None:
    print('Excluded support diagnostic is pending; no support verdict yet.')
print('These outputs are calibration and feasibility evidence; P1-P4 are untested.')


223c59d49fb1c2e3033630f941e5c2fff86b120e4855cac4ae6ae7222e394786 experiments/2026-10-20-c-law-renewal/results/calibration.json


60c51af722ba71a690fbe7290697f3fb72e9e6e2644fac48a6c45554e5ec3a4e experiments/2026-10-20-c-law-renewal/results/pilot_report.json
0c79950115ec834669cd9ede9a8c73846278caa250951a85bbc97d0ae78aa0d3 experiments/2026-10-20-c-law-renewal/results/pilot_diagnostics.json


aff83d63c41f98a45d30ddff3f4b13817084f397c2d3c8f537e4d54b78c8a553 experiments/2026-10-20-c-law-renewal/results/pilot_diagnostics.csv
Synthetic reward solver: maximum |c error| 0.0018502018274182408 signed bias 0.0002107888335104724 between-seed sd 0.000865310084626809
Hidden-history control: pooled c 0.499742 conditional contrast 0.8029957507452465
rotor c mean 0.4975039943500837 between-seed sd 0.00046687638082219373 escapes 0
rw c mean 0.44660243172590325 between-seed sd 0.008121941614407723 escapes 0
Original pilot horizon verdict: STOP_register_geometry_or_budget_revision
Usable k=26 runs: 2 / 24
Excluded support diagnostic is pending; no support verdict yet.
These outputs are calibration and feasibility evidence; P1-P4 are untested.


## Interpretation and next discrimination
The controls validate recording and the narrow solver task. They do not validate a model of ant state frequencies. The original pilot failed to supply a common sampling window; its support diagnostic cannot retroactively make that gate pass. Full eight-turn states may encode family identity and lose transfer support. A smaller state still needs a generative transition rule consistent with the target rule and a test against hidden history.
